# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahmarani/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [9]:
import duckdb
from google.colab import userdata

con = duckdb.connect()

con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{userdata.get('HF_TOKEN')}')")
base = "hf://datasets/FlyRank/internship-warehouse"
print("Setup done")

Setup done


In [10]:
src = f"read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')"
con.sql(f"DESCRIBE SELECT * FROM {src}").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [16]:
import pandas as pd, numpy as np

q = f"""
SELECT client_hash_id, content_hash_id,
  SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_h1,
  SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clicks_h1,
  AVG(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_avg_position END) AS pos_h1,
  SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN COALESCE(ga4_sessions, 0) ELSE 0 END) AS sessions_h1,
  COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15' AND gsc_impressions > 0 THEN report_date END) AS days_visible_h1,
  SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_h2
FROM {src}
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
HAVING SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) > 0
"""
feat = con.sql(q).df()
feat = feat.sample(min(60000, len(feat)), random_state=0).reset_index(drop=True)
feat["pos_h1"] = feat["pos_h1"].fillna(feat["pos_h1"].median())
# label (proxy): daily impressions fell in the second half (16 days) vs the first half (15 days)
feat["declined"] = ((feat.imp_h2 / 16) < (feat.imp_h1 / 15)).astype(int)
print(feat.shape, "| declined rate:", round(feat.declined.mean(), 3))
feat.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(60000, 9) | declined rate: 0.496


,client_hash_id,content_hash_id,imp_h1,clicks_h1,pos_h1,sessions_h1,days_visible_h1,imp_h2,declined
0,client_08a6a72ff48e62c0,content_6810888fe2a6739c,4.0,0.0,76.333333,0.0,2,53.0,0
1,client_3ffa76342f366962,content_ca32c2244270a0ef,1.0,0.0,11.000000,0.0,1,0.0,1
2,client_23a62021009f63c4,content_c475464411640300,18.0,0.0,1.740741,0.0,9,26.0,0
3,client_3f0ce4d44fe94f3d,content_d3da6b44a1dd7770,559.0,0.0,17.180829,0.0,15,940.0,0
4,client_def0955f7a377868,content_824ae6398a850550,15.0,1.0,14.000000,2.0,9,30.0,0


Five features, each knowable at the decision moment (the end of day 15 of the month):
- imp_h1: impressions in days 1-15, known once those days are reported.
- clicks_h1: clicks in days 1-15, same reason.
- pos_h1: average position in days 1-15, reported with the same daily data.
- sessions_h1: sessions in days 1-15, known after day 15.
- days_visible_h1: number of days in 1-15 with impressions, counted from those days only.
Label (proxy): daily impressions in days 16-end are lower than in days 1-15. The label uses only later days, so it never enters the features.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [15]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

def quick_auc(cols):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)
                  .split(feat, feat.declined, groups=feat.client_hash_id))
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    m.fit(feat.iloc[tr][cols], feat.iloc[tr].declined)
    return round(roc_auc_score(feat.iloc[te].declined, m.predict_proba(feat.iloc[te][cols])[:, 1]), 3)

honest_cols = ["imp_h1", "clicks_h1", "pos_h1", "sessions_h1", "days_visible_h1"]
print("Honest AUC (5 features, held-out clients):", quick_auc(honest_cols))

feat["leak_ratio"] = feat.imp_h2 / (feat.imp_h1 + 1)   # label-derived on purpose: uses the future window
print("AUC WITH the leaky column:", quick_auc(honest_cols + ["leak_ratio"]))

feat = feat.drop(columns=["leak_ratio"])               # remove the trap
print("AUC after deleting it (honest number):", quick_auc(honest_cols))

Honest AUC (5 features, held-out clients): 0.529
AUC WITH the leaky column: 1.0
AUC after deleting it (honest number): 0.529


The leakage hunt: I added leak_ratio = impressions in days 16-end divided by impressions in days 1-15. It is built from the same future days that define the label, so the quick score jumped from 0.522 to 1.0. I deleted it and kept the honest number, 0.522. The honest score is close to random (0.5), so these five early-window features alone say little about later decline.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [17]:
excluded = {
    "imp_h2 (days 16-end impressions)": "defines the label, so it is the future window",
    "leak_ratio": "built from the label window; made the score look perfect",
    "health_score, priority_score": "FlyRank decision flags that already encode its own judgment",
    "client_hash_id, content_hash_id": "used for grouping and joins only, not as features",
}
for k, v in excluded.items():
    print(f"{k}: {v}")

imp_h2 (days 16-end impressions): defines the label, so it is the future window
leak_ratio: built from the label window; made the score look perfect
health_score, priority_score: FlyRank decision flags that already encode its own judgment
client_hash_id, content_hash_id: used for grouping and joins only, not as features


Trap: I added leak_ratio = impressions in days 16-end divided by impressions in days 1-15. It is built from the same future days that define the label, so the quick score jumped from 0.522 to 1.0. I deleted it and kept the honest number, 0.522. The honest score is close to random (0.5), so these five early-window features alone say little about later decline; this is a data-contract exercise, not a model.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.